# Train XGBoost on Synthetic Phone Data

This notebook reproduces `train_xgboost.py`: synthetic data generation, training an XGBoost classifier, evaluation, and saving the model. Run cells in order and modify parameters in the config cell as needed.

In [ ]:
# Imports
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import os

print('Imports ready.')

In [ ]:
def generate_synthetic_data(num_samples=500, random_seed=42):
    import numpy as np
    import pandas as pd
    np.random.seed(random_seed)
    data = []
    classes = [0, 1, 2, 3]
    samples_per_class = num_samples // 4
    for cls in classes:
        for _ in range(samples_per_class):
            if cls == 0: # Reuse
                model_age_months = np.random.randint(1, 10)
                battery_health_pct = np.random.uniform(90.0, 100.0)
                screen_cracked = 0
                functional_issues = 0
                cosmetic_scratches = np.random.choice([0, 1], p=[0.9, 0.1])
            elif cls == 1: # Refurbish
                model_age_months = np.random.randint(8, 18)
                battery_health_pct = np.random.uniform(82.0, 92.0)
                screen_cracked = 0
                functional_issues = 0
                cosmetic_scratches = np.random.choice([0, 1, 2], p=[0.2, 0.7, 0.1])
            elif cls == 2: # Repair
                model_age_months = np.random.randint(6, 24)
                battery_health_pct = np.random.uniform(78.0, 90.0)
                screen_cracked = np.random.choice([0, 1], p=[0.3, 0.7])
                functional_issues = np.random.choice([0, 1], p=[0.4, 0.6])
                if screen_cracked == 0 and functional_issues == 0:
                    screen_cracked = 1
                cosmetic_scratches = np.random.choice([0, 1, 2], p=[0.3, 0.5, 0.2])
            else: # Recycle
                model_age_months = np.random.randint(20, 48)
                battery_health_pct = np.random.uniform(50.0, 79.9)
                screen_cracked = np.random.choice([0, 1], p=[0.5, 0.5])
                functional_issues = np.random.choice([0, 1], p=[0.2, 0.8])
                cosmetic_scratches = np.random.choice([0, 1, 2], p=[0.1, 0.3, 0.6])
            data.append({
                'model_age_months': model_age_months,
                'battery_health_pct': battery_health_pct,
                'screen_cracked': screen_cracked,
                'functional_issues': functional_issues,
                'cosmetic_scratches': cosmetic_scratches,
                'label': cls
            })
    df = pd.DataFrame(data)
    df = df.sample(frac=1.0, random_state=random_seed).reset_index(drop=True)
    return df

print('Synthetic data generator defined.')

In [ ]:
# Notebook configuration
params = {
    'num_samples': 600,
    'test_size': 0.2,
    'random_state': 42,
    'model_path': './xgboost_model.json'
}
print('Config set:', params)

In [ ]:
# Generate data and split
print('Generating synthetic data...')
df = generate_synthetic_data(num_samples=params['num_samples'], random_seed=params['random_state'])
X = df.drop(columns=['label'])
y = df['label']

print('\nDataset class distribution:')
print(y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=params['test_size'], random_state=params['random_state'])
print('\nSplit done. Train size:', X_train.shape[0], 'Test size:', X_test.shape[0])

In [ ]:
# Train XGBoost model
print('\nTraining XGBoost classifier...')
model = xgb.XGBClassifier(
    n_estimators=80,
    max_depth=4,
    learning_rate=0.1,
    random_state=params['random_state'],
    eval_metric='mlogloss'
)

model.fit(X_train, y_train)
print('Model trained.')

In [ ]:
# Evaluate
print('\nEvaluating model...')
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Validation Accuracy: {accuracy*100:.2f}%")

classes = ['Reuse', 'Refurbish', 'Repair', 'Recycle']
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=classes))

In [ ]:
# Save model
print('\nSaving model to', params['model_path'])
model.save_model(params['model_path'])
print('Model saved to:', os.path.abspath(params['model_path']))